In [ ]:
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
# plt.rcParams.update({
#     "font.size" : 18,
#     "font.family": "serif",
#     "font.serif": ["Times New Roman"],
#     "mathtext.fontset": "stix",
# })

In [ ]:

folderVec = [
    "/home/liangmj/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_25dx_test/probes/probe2",
]

rho0 = 1.0
cs2 = 1.0 / 3.0  # lattice speed of sound squared

U0Vec = [
    0.115470054,
]

startStep = 40000

avgStepRg = [
    [70000, 100000],
]

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    '#FF5733',
    "#119100",
    "#B700FF",
]

labelVec = [
    # "Present LBM",
    "LBM Cumulant D/dx=25",
]

In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordsStr = header.split('(')[1].split(')')[0].split(',')
        x = float(coordsStr[0])
        y = float(coordsStr[1])
        z = float(coordsStr[2])
    return x,y,z

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec


In [ ]:

thetaVecVec = []
pCoefVecVec = []
for case in range(0, len(folderVec)):
    files = sorted(glob.glob(os.path.join(folderVec[case], "probe*.txt")))
    nProbes = len(files)

    stepCol = read_col_probe(files[0], 0, 2)
    idxStart = int(np.abs(stepCol - avgStepRg[case][0]).argmin())
    idxEnd = int(np.abs(stepCol - avgStepRg[case][1]).argmin())
    thetaVec = []
    pCoefVec = []
    dynPressure = 0.5 * rho0 * U0Vec[case]**2
    for iProbe in range(0, len(files)):
        x, y, z = get_probe_coords(files[iProbe])
        theta = np.degrees(np.arctan2(y, x)) + 180
        thetaVec.append(theta)
        rhoCol = read_col_probe(files[iProbe], 2, 2)
        pressure = (rhoCol - rho0) * cs2
        avgPressure = np.mean(pressure[idxStart:idxEnd])
        pCoef = avgPressure / dynPressure
        pCoefVec.append(pCoef)
    thetaVecVec.append(thetaVec)
    pCoefVecVec.append(pCoefVec)
    

In [ ]:
refCpFile = "/home/liangmj/repo/scripts/cases/cylinder_Re150_acoustic/ref/cp_inoue_Ma02.csv"
refTheta = read_csv_col(refCpFile, 0, 1)
refCp = read_csv_col(refCpFile, 1, 1)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6), facecolor='w', edgecolor='w')

ax.plot(refTheta, refCp, lw=2, c='k', label='Inoue & Hatakeyama')
for case in range(0, len(folderVec)):
    ax.plot(thetaVecVec[case][:90], pCoefVecVec[case][:90], c=colorVec[case], lw=0, marker='o', fillstyle='none', ms=6, mew=2, markevery=2, label=labelVec[case])

ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$C_p$")
ax.set_xlim(0, 180)
ax.set_ylim(-1.5, 1.5)
ax.tick_params(width=2.0, axis='both', direction='in')
for spine in ax.spines.values():
    spine.set_linewidth(2.0)

ax.legend(loc='upper right', frameon=False, fontsize=15)